# Reasoning Prompt Patterns — Hands-On

**LLM Engineering · Domain 3 · Roadmap Week 15**

Offline notebook: deterministic fake models demonstrate CoT, ReAct, self-consistency, decomposition, and reflection control flow.

## 0. Setup: tiny arithmetic task

In [ ]:
%pip install -q numpy
from collections import Counter
import re

question = "What is 18 * (7 + 5)?"
def safe_calc(expr):
    if not set(expr) <= set("0123456789+-*/() "):
        raise ValueError("unsafe")
    return eval(expr, {"__builtins__": {}}, {})
print(question, "=>", safe_calc("18 * (7 + 5)"))

## 1. Direct vs chain-of-thought simulation

In [ ]:
def direct_answer(q):
    return 18 * 7 + 5  # common precedence mistake

def scratchpad_answer(q):
    steps = ["compute parentheses: 7 + 5 = 12", "multiply: 18 * 12 = 216"]
    return 216, steps

print("direct:", direct_answer(question))
ans, steps = scratchpad_answer(question)
print("scratchpad final:", ans)
for s in steps: print(" -", s)

## 2. ReAct: reason, call a tool, observe, answer

In [ ]:
def react(q):
    expr = re.search(r"What is (.*)\?", q).group(1)
    transcript = []
    transcript.append(("Thought", "Need exact arithmetic, so call calculator."))
    transcript.append(("Action", f"calculator({expr!r})"))
    obs = safe_calc(expr)
    transcript.append(("Observation", str(obs)))
    transcript.append(("Final", f"The answer is {obs}."))
    return transcript

for k, v in react(question): print(f"{k:>11}: {v}")

## 3. Self-consistency: vote across noisy reasoning paths

In [ ]:
def noisy_path(seed):
    # Two out of three paths are correct; every third path makes a slip.
    return 217 if seed % 3 == 0 else 216

def self_consistency(samples):
    answers = [noisy_path(s) for s in range(samples)]
    vote, count = Counter(answers).most_common(1)[0]
    return vote, count, answers

for n in [1, 3, 7, 11]:
    vote, count, answers = self_consistency(n)
    print(f"samples={n:<2} vote={vote} count={count} answers={answers}")

## 4. Least-to-most decomposition

In [ ]:
def decompose(q):
    return ["What is 7 + 5?", "What is 18 times that result?"]

subanswers = []
for subq in decompose(question):
    if "7 + 5" in subq: subanswers.append(12)
    else: subanswers.append(18 * subanswers[-1])
print("subquestions:", decompose(question))
print("subanswers:", subanswers, "final", subanswers[-1])

## 5. Reflection grounded in a verifier

In [ ]:
def verifier(candidate):
    return candidate == safe_calc("18 * (7 + 5)")

candidate = direct_answer(question)
print("candidate", candidate, "passes verifier?", verifier(candidate))
if not verifier(candidate):
    candidate, _ = scratchpad_answer(question)
print("revised", candidate, "passes verifier?", verifier(candidate))

## 6. Exercises
1. Add a max-iteration guard to the ReAct loop.
2. Change noisy_path so the majority is wrong; what does that teach about voting?
3. Add a cost model: `cost = samples * tokens_per_path`.
4. Route tasks: direct if easy, ReAct if a tool is needed, self-consistency if confidence is low.

## Links
- Literature note: `02 Literature Notes/LLM Engineering/Reasoning Prompt Patterns`
- Snippets: `04 Code Snippets/LLM/Self Consistency Voting Simulator`, `.../ReAct Loop With Fake Tools`
- MOC: `06 Maps of Content/LLM Engineering Concepts`